# 08 — Hierarchical Models
**References:** Gelman et al. BDA3 Ch. 5 · McElreath (2020) Ch. 13–14 · Gelman & Hill (2007)

## Narrative thread
```
Complete vs no-pooling -> Partial pooling -> Hierarchical structure -> Hyperpriors -> Shrinkage -> 8 Schools
```

## 1. The pooling spectrum

Consider $J$ groups, each with $n_j$ observations $y_{ij} \sim \mathcal{N}(\theta_j, \sigma^2)$.

| Strategy | Model | Problem |
|---|---|---|
| **No pooling** | Estimate $\theta_j$ independently | High variance for small $n_j$; ignores information sharing |
| **Complete pooling** | $\theta_j = \theta$ for all $j$ | Ignores group differences; biased |
| **Partial pooling** | $\theta_j \sim \mathcal{N}(\mu, \tau^2)$ | Best of both: borrows strength across groups |

**Hierarchical model:**
$$y_{ij} \mid \theta_j, \sigma \sim \mathcal{N}(\theta_j, \sigma^2) \qquad \text{(likelihood)}$$
$$\theta_j \mid \mu, \tau \sim \mathcal{N}(\mu, \tau^2) \qquad \text{(group-level model)}$$
$$\mu \sim p(\mu), \quad \tau \sim p(\tau) \qquad \text{(hyperpriors)}$$

**Shrinkage:** The posterior mean of $\theta_j$ is pulled toward the grand mean $\mu$:
$$E[\theta_j \mid \mathbf{y}] \approx \frac{n_j/\sigma^2}{n_j/\sigma^2 + 1/\tau^2}\bar y_j + \frac{1/\tau^2}{n_j/\sigma^2 + 1/\tau^2}\mu$$

Groups with small $n_j$ are shrunk more toward $\mu$; groups with large $n_j$ rely more on their own data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── 8 Schools: the canonical hierarchical model example ───────────────────
# Rubin (1981): SAT coaching programs at 8 schools
# y_j: estimated effect of coaching, sigma_j: known SE
y_schools     = np.array([28, 8, -3, 7, -1, 1, 18, 12], dtype=float)
sigma_schools = np.array([15, 10, 16, 11, 9, 11, 10, 18], dtype=float)
school_labels = ['A','B','C','D','E','F','G','H']
J = len(y_schools)

# Analytic partial pooling (known tau approach for illustration)
# For a fixed tau, posterior mean of theta_j:
#   mu_hat = weighted mean = sum(y_j/V_j) / sum(1/V_j)
#   where V_j = sigma_j^2 + tau^2
#   post_mean_j = (y_j/sigma_j^2 + mu_hat/tau^2) / (1/sigma_j^2 + 1/tau^2)

def partial_pool_estimates(y, sigma, tau):
    V    = sigma**2 + tau**2
    mu   = np.sum(y / V) / np.sum(1 / V)
    prec = 1/sigma**2 + 1/tau**2
    theta_post = (y/sigma**2 + mu/tau**2) / prec
    return theta_post, mu

# Compare pooling strategies for different tau values
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
x_pos = np.arange(J)

for ax, tau, title in zip(axes, [1, 10, 100],
                            ['$\\tau=1$: near complete pooling', '$\\tau=10$: moderate partial pooling', '$\\tau=100$: near no pooling']):
    pp, mu_hat = partial_pool_estimates(y_schools, sigma_schools, tau)
    ax.errorbar(x_pos, y_schools, yerr=1.96*sigma_schools, fmt='o', color='#f72585',
                capsize=5, markersize=7, label='No pooling (raw $\\pm$ 95% CI)')
    ax.plot(x_pos, pp, 's', color='#4361ee', markersize=9, label='Partial pooling')
    ax.axhline(mu_hat, color='#06d6a0', lw=2, linestyle='--', label=f'Grand mean $\\hat\\mu={mu_hat:.1f}$')
    ax.set_xticks(x_pos); ax.set_xticklabels(school_labels)
    ax.set_xlabel('School'); ax.set_ylabel('Effect estimate')
    ax.set_title(title)
    ax.legend(fontsize=7)

plt.suptitle('8 Schools: Partial Pooling under Different τ Values\n'
             'Small τ → strong pooling toward grand mean; large τ → near no pooling',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Gibbs sampler for the 8 Schools model

Full conditionals for the normal hierarchical model:

$$\theta_j \mid \mu, \tau, y_j \sim \mathcal{N}\!\left(\frac{y_j/\sigma_j^2 + \mu/\tau^2}{1/\sigma_j^2 + 1/\tau^2},\; \frac{1}{1/\sigma_j^2 + 1/\tau^2}\right)$$

$$\mu \mid \boldsymbol{\theta}, \tau \sim \mathcal{N}\!\left(\bar\theta,\; \tau^2/J\right)$$

$$\tau^2 \mid \boldsymbol{\theta}, \mu \sim \text{IG}\!\left(J/2,\; \sum_j(\theta_j - \mu)^2/2\right) \quad \text{(with }p(\tau)\propto 1\text{ prior)}$$

In [ ]:
# ── Gibbs sampler for 8 Schools ────────────────────────────────────────────
def gibbs_8schools(y, sigma, n_iter=10_000, warmup=2000):
    J     = len(y)
    mu    = y.mean(); tau = 10.0
    theta = y.copy()
    chain = {'theta': [], 'mu': [], 'tau': []}

    for _ in range(n_iter):
        # Sample theta_j
        prec_j  = 1/sigma**2 + 1/tau**2
        mean_j  = (y/sigma**2 + mu/tau**2) / prec_j
        theta   = np.random.normal(mean_j, 1/np.sqrt(prec_j))

        # Sample mu
        mu = np.random.normal(theta.mean(), tau/np.sqrt(J))

        # Sample tau^2 from IG(J/2, SS/2)
        SS    = np.sum((theta - mu)**2)
        tau2  = 1/np.random.gamma(J/2, 1/(SS/2 + 1e-6))
        tau   = np.sqrt(max(tau2, 1e-6))

        chain['theta'].append(theta.copy())
        chain['mu'].append(mu)
        chain['tau'].append(tau)

    return {k: np.array(v[warmup:]) for k, v in chain.items()}

posterior = gibbs_8schools(y_schools, sigma_schools)
theta_post = posterior['theta']  # shape (n_samples, 8)
mu_post    = posterior['mu']
tau_post   = posterior['tau']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Posterior distributions for each school
ax0 = axes[0]
colors_schools = plt.cm.tab10(np.linspace(0, 1, J))
for j, (label, c) in enumerate(zip(school_labels, colors_schools)):
    ax0.violinplot(theta_post[:, j], positions=[j], showmedians=True)
ax0.scatter(range(J), y_schools, color='#f72585', s=80, zorder=5, label='Raw estimate $y_j$')
ax0.axhline(mu_post.mean(), color='#06d6a0', lw=2, linestyle='--', label=f'Post. mean $\\mu$={mu_post.mean():.1f}')
ax0.set_xticks(range(J)); ax0.set_xticklabels(school_labels)
ax0.set_title('Posterior $\\theta_j$ by school\nBlue violins = posterior; Red = raw data')
ax0.legend(fontsize=8)

# Posterior of tau (between-school heterogeneity)
ax1 = axes[1]
ax1.hist(tau_post, bins=60, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
ax1.axvline(np.median(tau_post), color='#f72585', lw=2, linestyle='--',
            label=f'Median $\\tau$={np.median(tau_post):.1f}')
ax1.set_xlabel('$\\tau$ (between-school heterogeneity)')
ax1.set_title('Posterior of $\\tau$\nHigh uncertainty about between-school variance')
ax1.legend()

# Shrinkage plot
ax2 = axes[2]
theta_pm = theta_post.mean(axis=0)
for j, label in enumerate(school_labels):
    ax2.annotate('', xy=(theta_pm[j], j), xytext=(y_schools[j], j),
                 arrowprops=dict(arrowstyle='->', color='#4361ee', lw=2))
ax2.scatter(y_schools, range(J), color='#f72585', s=80, zorder=5, label='Raw $y_j$')
ax2.scatter(theta_pm,  range(J), color='#4361ee', s=80, zorder=5, label='Post. mean $\\theta_j$')
ax2.axvline(mu_post.mean(), color='#06d6a0', lw=2, linestyle='--', label=f'Grand mean')
ax2.set_yticks(range(J)); ax2.set_yticklabels(school_labels)
ax2.set_xlabel('Estimated effect'); ax2.set_title('Shrinkage: raw → posterior mean\nAll estimates pulled toward grand mean')
ax2.legend(fontsize=8)

plt.suptitle('8 Schools Hierarchical Model: Gibbs Sampler Results', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Key takeaways

| Concept | Statement |
|---|---|
| Partial pooling | Borrows strength across groups — optimal between no- and complete-pooling |
| Shrinkage factor | $n_j/\sigma^2 / (n_j/\sigma^2 + 1/\tau^2)$ — small groups shrink more |
| Hyperpriors on $\tau$ | Let data determine amount of pooling — more Bayesian than EB |
| Full conditionals | Known distributions → Gibbs sampler; otherwise use MH or HMC |
| Generalization | Any multilevel structure: schools within districts, patients within hospitals |

**Next:** notebook 09 — Bayesian regression: priors on coefficients, variable selection, horseshoe prior.